# 🛍️ E-Commerce Sales Analysis — Indonesia 2023

**Tools:** Python, Pandas, Matplotlib, Seaborn  
**Dataset:** Synthetic e-commerce transaction data (2,000 orders)  
**Objective:** Analyze sales performance, customer behavior, and business trends to generate actionable insights.

---

## 1. Import Libraries & Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('ecommerce_sales_2023.csv', parse_dates=['date'])
df['month']   = df['date'].dt.to_period('M').astype(str)
df['month_n'] = df['date'].dt.month
df['quarter'] = df['date'].dt.quarter.map({1:'Q1',2:'Q2',3:'Q3',4:'Q4'})

print('Dataset shape:', df.shape)
df.head()

## 2. Data Overview & Quality Check

In [ ]:
print('=== DATA INFO ===')
print(df.dtypes)
print('\n=== MISSING VALUES ===')
print(df.isnull().sum())
print('\n=== DUPLICATES ===')
print(f'Duplicate rows: {df.duplicated().sum()}')

In [ ]:
print('=== DESCRIPTIVE STATISTICS ===')
df[['unit_price', 'quantity', 'revenue']].describe().applymap(lambda x: f'{x:,.0f}')

## 3. Summary KPIs

In [ ]:
completed = df[df['status'] == 'Completed']

total_orders    = len(df)
completed_orders= len(completed)
total_revenue   = completed['revenue'].sum()
avg_order_value = completed['revenue'].mean()
cancel_rate     = (df['status'] == 'Cancelled').mean() * 100

print(f'📦 Total Orders      : {total_orders:,}')
print(f'✅ Completed Orders  : {completed_orders:,} ({completed_orders/total_orders*100:.1f}%)')
print(f'💰 Total Revenue     : Rp{total_revenue:,.0f}')
print(f'🛒 Avg Order Value   : Rp{avg_order_value:,.0f}')
print(f'❌ Cancellation Rate : {cancel_rate:.1f}%')

## 4. Monthly Revenue Trend

In [ ]:
BLUE = '#185FA5'; LIGHT = '#B5D4F4'; GRAY = '#888780'; ACCENT = '#E24B4A'; TEXT = '#1a1a1a'

plt.rcParams.update({
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25, 'grid.linestyle': '--',
    'axes.facecolor': '#F8F9FA', 'figure.facecolor': 'white',
})

def fmt_idr(x, _=None):
    if x >= 1e9: return f'Rp{x/1e9:.1f}B'
    if x >= 1e6: return f'Rp{x/1e6:.0f}M'
    return f'Rp{x:,.0f}'

monthly = completed.groupby('month')['revenue'].sum().reset_index().sort_values('month')

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.fill_between(range(len(monthly)), monthly['revenue'], alpha=0.12, color=BLUE)
ax.plot(range(len(monthly)), monthly['revenue'], color=BLUE, linewidth=2.5, marker='o', markersize=5)

peak_idx = monthly['revenue'].idxmax()
ax.annotate(f"Peak\n{fmt_idr(monthly['revenue'].max())}",
            xy=(list(monthly.index).index(peak_idx), monthly['revenue'].max()),
            xytext=(list(monthly.index).index(peak_idx)-2, monthly['revenue'].max()*0.92),
            arrowprops=dict(arrowstyle='->', color=ACCENT, lw=1.5),
            fontsize=9, color=ACCENT, fontweight='bold')

ax.set_xticks(range(len(monthly)))
ax.set_xticklabels([m[-5:] for m in monthly['month']], rotation=30, ha='right', fontsize=8)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_idr))
ax.set_title('Monthly Revenue Trend — 2023', fontsize=13, fontweight='bold')
ax.set_xlabel('Month'); ax.set_ylabel('Revenue')
plt.tight_layout()
plt.savefig('fig1_monthly_revenue.png', dpi=150, bbox_inches='tight')
plt.show()

**Insight:** Revenue fluctuates monthly with a notable peak in Q3. Seasonal promotions (Harbolnas, Ramadan) likely drive spikes.

## 5. Category Analysis

In [ ]:
cat_rev = completed.groupby('category')['revenue'].sum().sort_values(ascending=True)
cat_ord = df.groupby('category')['order_id'].count().reindex(cat_rev.index)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Revenue
colors_bar = [BLUE if c == cat_rev.index[-1] else LIGHT for c in cat_rev.index]
bars = axes[0].barh(cat_rev.index, cat_rev.values, color=colors_bar, height=0.6, edgecolor='none')
for bar, val in zip(bars, cat_rev.values):
    axes[0].text(val + cat_rev.max()*0.01, bar.get_y()+bar.get_height()/2,
                 fmt_idr(val), va='center', fontsize=8)
axes[0].set_title('Revenue by Category', fontweight='bold')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_idr))

# Orders
colors_bar2 = [BLUE if c == cat_ord.index[-1] else LIGHT for c in cat_ord.index]
bars2 = axes[1].barh(cat_ord.index, cat_ord.values, color=colors_bar2, height=0.6, edgecolor='none')
for bar, val in zip(bars2, cat_ord.values):
    axes[1].text(val + cat_ord.max()*0.01, bar.get_y()+bar.get_height()/2,
                 str(val), va='center', fontsize=8)
axes[1].set_title('Total Orders by Category', fontweight='bold')

plt.tight_layout()
plt.savefig('fig2_category_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

**Insight:** Electronics leads in revenue despite fewer orders, due to high unit price. Fashion has the highest order volume — high demand, lower margin per item.

## 6. City & Payment Method Analysis

In [ ]:
city_rev = completed.groupby('city')['revenue'].sum().sort_values(ascending=False)
pay_dist = df.groupby('payment_method')['order_id'].count()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# City
pal = ['#185FA5','#378ADD','#85B7EB','#B5D4F4','#D0E6F7','#E3F0FA','#EFF6FD']
axes[0].bar(city_rev.index, city_rev.values, color=pal[:len(city_rev)], edgecolor='none', width=0.6)
for i, (city, val) in enumerate(city_rev.items()):
    axes[0].text(i, val + city_rev.max()*0.01, fmt_idr(val), ha='center', fontsize=7.5)
axes[0].set_title('Revenue by City', fontweight='bold')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_idr))
axes[0].tick_params(axis='x', rotation=20)

# Payment
wedge_colors = ['#185FA5','#378ADD','#85B7EB','#B5D4F4']
wedges, texts, autotexts = axes[1].pie(
    pay_dist.values, labels=pay_dist.index, autopct='%1.1f%%',
    colors=wedge_colors, startangle=140,
    wedgeprops=dict(edgecolor='white', linewidth=2), textprops={'fontsize': 9})
for at in autotexts: at.set_color('white'); at.set_fontweight('bold')
axes[1].set_title('Payment Method Distribution', fontweight='bold')

plt.tight_layout()
plt.savefig('fig3_city_payment.png', dpi=150, bbox_inches='tight')
plt.show()

**Insight:** Jakarta dominates revenue (35% of total), consistent with its population share. E-Wallet and Credit Card together account for ~58% of transactions, indicating strong digital payment adoption.

## 7. Order Status & Quarterly Revenue

In [ ]:
status_cnt = df['status'].value_counts()
q_rev = completed.groupby('quarter')['revenue'].sum()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

s_colors = {'Completed': '#3B6D11', 'Cancelled': '#E24B4A', 'Returned': '#BA7517'}
axes[0].bar(status_cnt.index, status_cnt.values,
            color=[s_colors[s] for s in status_cnt.index], edgecolor='none', width=0.5)
for i, (s, v) in enumerate(status_cnt.items()):
    axes[0].text(i, v + 10, str(v), ha='center', fontsize=10, fontweight='bold')
axes[0].set_title('Order Status Distribution', fontweight='bold')
axes[0].set_ylabel('Number of Orders')

q_colors = [BLUE if q == q_rev.idxmax() else LIGHT for q in q_rev.index]
axes[1].bar(q_rev.index, q_rev.values, color=q_colors, edgecolor='none', width=0.5)
for i, (q, v) in enumerate(q_rev.items()):
    axes[1].text(i, v + q_rev.max()*0.01, fmt_idr(v), ha='center', fontsize=9)
axes[1].set_title('Revenue by Quarter', fontweight='bold')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_idr))

plt.tight_layout()
plt.savefig('fig4_status_quarterly.png', dpi=150, bbox_inches='tight')
plt.show()

**Insight:** 79.6% completion rate is healthy. Q3 is the strongest quarter — likely driven by mid-year sales events. Cancellation rate of 12.8% warrants further investigation into checkout UX or payment failures.

## 8. Key Findings & Business Recommendations

| # | Finding | Recommendation |
|---|---------|----------------|
| 1 | Electronics generates the highest revenue (high unit price) | Prioritize stock availability & flash sales for Electronics |
| 2 | Fashion has the most orders but lower margin | Bundle deals or upselling strategy for Fashion |
| 3 | Jakarta dominates — other cities underperform | Expand marketing budget to Surabaya & Bandung |
| 4 | E-Wallet is top payment method (28%) | Offer cashback promotions for E-Wallet users |
| 5 | Q3 is peak quarter | Plan inventory & logistics ahead of Q3 |
| 6 | 12.8% cancellation rate | Investigate checkout drop-off; improve payment flow |

---
*Analysis by Yusril Arbizal — github.com/zRILLL28*